# Explore low-lewel data in HDF5 format

### Check paths and environments

In [ ]:
# Where are we?
! pwd

In [ ]:
# Check for the versions of the core dependencies 
! conda list | grep ctlearn
! conda list | grep astropy
! conda list | grep ctapipe
! conda list | grep dl1-data-handler 
! conda list | grep keras
! conda list | grep tensorflow

In [ ]:
# Set here correct path to downloaded test data of CTAO simulation.
DATA_DIR = "../../../../iaa-advanced-neural-networks-2026-ctao-data"
! du -h {DATA_DIR}/hdf5_merged/*

### Browsing through the HDF5 files via vitables
Use ViTables, a convenient GUI, to explore the data model.

In [ ]:
#! conda run -n vitables vitables {DATA_DIR}/hdf5_merged/*

### Let's import some plotting libraries

In [ ]:
import numpy as np
import astropy.units as u
import numpy as np
from scipy.stats import norm
from matplotlib import pyplot as plt
import matplotlib.animation as ani
from IPython.display import HTML

# For the ctapipe section
from ctapipe.image import hillas_parameters
from ctapipe.io import EventSource
from ctapipe.visualization import CameraDisplay

# For the DL1DH section
from dl1_data_handler.reader import DLImageReader, DLWaveformReader


# %matplotlib inline
plt.style.use("ggplot")

## Exploring the data files with ctapipe
Besides vitables we can also explore the data with the reading functionality of the ctapipe.

https://ctapipe.readthedocs.io/en/latest/auto_examples/tutorials/raw_data_exploration.html
https://ctapipe.readthedocs.io/en/latest/auto_examples/tutorials/calibrated_data_exploration.html

In [ ]:
# Explore gamma-ray events
filename = f"{DATA_DIR}/hdf5_merged/gamma_theta_16.087_az_108.090_runs1-2.r1.dl1.h5"
# Uncomment to explore proton events
#filename = f"{DATA_DIR}/hdf5_merged/proton_theta_16.087_az_108.090_runs1-2.r1.dl1.h5"


source = EventSource(filename, max_events=5)
# One can loop over the 'EventSource' or use 'iter()' and 'next()'. It is a generator!
event_iterator = iter(source)
event = next(event_iterator)
# Uncomment to get second event (less beautiful)
#event = next(event_iterator)

# Print the event data structure
print(event)
# One can print the simulated energy which is an astropy quantity
print(f"MC energy in TeV: {event.simulation.shower.energy}")
# Convert from TeV to GeV
print(f"MC energy in GeV: {event.simulation.shower.energy.to('GeV')}")

# Same for the simulated arrival direction, e.g. altitude
print("Altitude in degrees:", event.simulation.shower.alt)
print("Altitude in radians:", event.simulation.shower.alt.to('rad'))

# Can be also hardcoded to 'tel_id = 1' since we have LST-1 simulations
tel_id = sorted(event.r1.tel.keys())[0]

# Print the description of the subarray
print(source.subarray.info())

# Store plotting data in convienent variables
geometry = source.subarray.tel[tel_id].camera.geometry
calibrated_waveform = event.r1.tel[tel_id].waveform[0] # 
image = event.dl1.tel[tel_id].image
peak_time = event.dl1.tel[tel_id].peak_time
image_mask = event.dl1.tel[tel_id].image_mask
image_parameters = event.dl1.tel[tel_id].parameters.hillas
cleaned_image = image.copy()
cleaned_image[~image_mask] = 0
cleaned_peak_time = peak_time.copy()
cleaned_peak_time[~image_mask] = 0
signal_pixels = np.nonzero(image_mask)[0]
background_pixels = np.nonzero(~image_mask)[0]

### Explore calibrated waveforms

#### Plot waveforms of the pixels

In [ ]:
plt.pcolormesh(calibrated_waveform)
# Note that after selecting specific pixels, the y-axis don not match pixel_ids anymore
#plt.pcolormesh(calibrated_waveform[signal_pixels])
#plt.pcolormesh(calibrated_waveform[background_pixels])
plt.colorbar()
plt.xlabel("sample number in ns")
plt.ylabel("Pixel_id")

#### Plot the waveforms of signal/background pixels in the readout window

In [ ]:
selected_pixels = signal_pixels
# Uncomment for plotting the calibrated waveform of background pixels
#selected_pixels = background_pixels
fig, (ax1, ax2) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(8, 6),
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05},
)

# Top: calibrated waveforms
for pix_id in selected_pixels:
    ax1.plot(calibrated_waveform[pix_id], drawstyle="steps")

ax1.set_ylabel("p.e.")

# Bottom: Gaussian of peak arrival times
mu, sigma = norm.fit(peak_time[selected_pixels])
x = np.linspace(0, calibrated_waveform.shape[1] - 1, 500)
y = norm.pdf(x, mu, sigma)

ax2.plot(
    x, y,
    color="C3",
    lw=2,
    label=rf"$\mu={mu:.2f}$ ns, $\sigma={sigma:.2f}$ ns"
)
ax2.fill_between(x, y, color="C3", alpha=0.3)

ax2.set_xlabel("sample number (ns)")
ax2.set_ylabel("Gaussian")
ax2.legend(loc="upper right", frameon=False)

plt.show()

#### Plot the shower development in the camera

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# First frame
disp = CameraDisplay(
    geometry,
    image=calibrated_waveform[:, 0],
    ax=ax,
    norm="lin",
)

disp.add_colorbar(ax=ax)
disp.set_limits_minmax(
    calibrated_waveform.min(),
    calibrated_waveform.max(),
)

time_text = ax.set_title("t = 0 ns")

def update(frame):
    disp.image = calibrated_waveform[:, frame]
    time_text.set_text(f"t = {frame} ns")
    return disp.pixels, time_text

anim = ani.FuncAnimation(
    fig,
    update,
    frames=calibrated_waveform.shape[1],
    interval=500,      # ms between frames
    blit=False,
)

plt.close(fig)

HTML(anim.to_jshtml())

# Uncomment to save as GIF and show it your friends!
#anim.save("camera_waveform.gif", writer=ani.PillowWriter(fps=500))

### Plot event images (charges or peak arrival times)

In [ ]:
disp = CameraDisplay(geometry, image=image, norm="lin")
disp.add_colorbar(label="p.e.")
# Uncomment to plot peak arrival time in nanoseconds
#disp = CameraDisplay(geometry, image=peak_time, norm="lin")
#disp.add_colorbar(label="ns")

### Plot cleaned images (charges or peak arrival times)

In [ ]:
disp_cleaned = CameraDisplay(geometry, image=cleaned_image, norm="log")
disp_cleaned.add_colorbar(label="p.e.")
# Uncomment to plot peak arrival time in nanoseconds
#disp = CameraDisplay(geometry, image=cleaned_peak_time, norm="lin")
#disp.add_colorbar(label="ns")

### Plot shower with hillas parametrization
For a more sophisticated event display with hillas parameters check out:
https://ctapipe.readthedocs.io/en/latest/auto_examples/visualization/hillas.html

In [ ]:
# We recalculate the hillas parameters in the camera frame (in meters), so it is easier to plot with the camera geometry!
image_parameters_camera_frame = hillas_parameters(geometry, cleaned_image)
print(f"Parameters in camera: {image_parameters_camera_frame}")
print(f"Parameters in sky (from hdf5 file): {image_parameters}")
plt.figure(figsize=(10, 10))
disp_hillas = CameraDisplay(geometry, image=image)
disp_hillas.add_colorbar(label="p.e.")
disp_hillas.overlay_moments(image_parameters_camera_frame, color="red", lw=3)
disp_hillas.highlight_pixels(image_mask, color="blue", alpha=0.3, linewidth=2)

plt.xlim(image_parameters_camera_frame.x.to_value(u.m) - 0.5, image_parameters_camera_frame.x.to_value(u.m) + 0.5)
plt.ylim(image_parameters_camera_frame.y.to_value(u.m) - 0.5, image_parameters_camera_frame.y.to_value(u.m) + 0.5)


## Add here ctapipe TableLoader

## Exploring the data files with DL1DH
Besides vitables we can also explore the data with the reading functionality of the DL1DH. This is a cruical part to inspect the previously performed data reduction through ctapipe. With this notebook one can spot corrupted data and ensure that the input provided to CTLearn's CNNs is correct.

In [ ]:
LST1_gammafile = f"{DATA_DIR}/hdf5_merged/gamma_theta_16.087_az_108.090_runs1-2.r1.dl1.h5"
LST1_protonfile = f"{DATA_DIR}/hdf5_merged/proton_theta_16.087_az_108.090_runs1-2.r1.dl1.h5"

### Image reading
The basic functionality of reading images for a telescope operating in monoscopic mode. 

In [ ]:
mono_reader = DLImageReader(
    input_url_signal=[LST1_gammafile],
    input_url_background=[LST1_protonfile],
)

print(mono_reader)
print(mono_reader.n_signal_events)
print(mono_reader.n_bkg_events)
print(mono_reader.subarray.info())
mono_reader.simulation_info.show_in_notebook(
    backend="classic",
    display_length=-1
)
# Add more content


### Waveform reading
The basic functionality of reading waveforms for a telescope operating in monoscopic mode.

In [ ]:
waveform_mono_reader = DLWaveformReader(
    input_url_signal=[LST1_gammafile],
    input_url_background=[LST1_protonfile],
)
# Add more content

### Stereo reading
The basic functionality of reading images for a telescope system operating in stereoscopic mode.

In [ ]:
# Stereo reading
# Add more content here